<a href="https://colab.research.google.com/github/winston-k/Wi20260413/blob/main/py2251h_ann110.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Artificial neural network example,
#  classification, diabetes dataset

In [16]:
# first neural network with keras tutorial
from numpy import loadtxt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [17]:
from google.colab import drive

drive.mount('/content/drive')

# Load the dataset
fp = "/content/drive/My Drive/Emeritus/Wi20260418/Module2-9/datasets_228_482_diabetes_nh.csv"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# load the dataset
dataset = loadtxt(fp, delimiter=',')
# split into input (X) and output (y) variables
Xtr = dataset[:500,0:8]
ytr = dataset[:500,8]
Xtst = dataset[501:768,0:8]
ytst = dataset[501:768,8]


In [19]:
# define the keras model
model = Sequential()
model.add(Dense(20, input_dim=8, activation='relu'))
model.add(Dense(20, activation='relu'))
model.add(Dense(20, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model = Sequential([
    keras.Input(shape=(8,)), # Define input shape using Input layer
    Dense(20, activation='relu'),
    Dense(20, activation='relu'),
    Dense(20, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 20)             │           180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,041 (4.07 KB)

 Trainable params: 1,041 (4.07 KB)

 Non-trainable params: 0 (0.00 B)

### Batch and Batch Size

In the context of training neural networks, 'batch' and 'batch size' refer to how your training data is processed during each update of the model's weights:

*   **Batch**: Instead of feeding the entire dataset to the neural network at once (which can be computationally expensive and memory-intensive for large datasets), the training data is divided into smaller subsets called batches. The model's weights are updated after processing each batch.

*   **Batch Size**: This is the number of training examples included in a single batch. For instance, in your `model.fit(Xtr, ytr, epochs=300, batch_size=10)` command, `batch_size=10` means that the model will process 10 training examples at a time, calculate the loss for those 10 examples, and then update its weights based on that batch's loss before moving to the next batch of 10 examples. This process repeats until all training examples have been seen once (which completes one epoch).

**Why use batches?**
*   **Computational Efficiency**: Smaller batches require less memory, allowing for larger datasets to be trained even on hardware with limited resources.
*   **Regularization**: Training with small batches introduces some noise in the gradient updates, which can act as a regularizer, potentially leading to better generalization and preventing the model from getting stuck in sharp local minima.
*   **Faster Convergence (sometimes)**: For very large datasets, smaller batches can lead to quicker updates and faster convergence in some cases, though this depends on the specific problem and model architecture.

In [27]:
# compile the keras model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# fit the keras model on the dataset
model.fit(Xtr, ytr, epochs=5000, batch_size=10, verbose=0)
# evaluate the keras model
_, accuracy = model.evaluate(Xtst, ytst, verbose=0)
print('Accuracy: %.2f ' % (accuracy*100))

Accuracy: 72.28 


In [22]:
# testing the model
y_pred = model.predict(Xtst)
y_pred = np.round(y_pred)
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(ytst, y_pred))
print(classification_report(ytst, y_pred))

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[163  18]
 [ 50  36]]
              precision    recall  f1-score   support

         0.0       0.77      0.90      0.83       181
         1.0       0.67      0.42      0.51        86

    accuracy                           0.75       267
   macro avg       0.72      0.66      0.67       267
weighted avg       0.73      0.75      0.73       267



## Practices ...
1. Change the runtime type A100 GPU, what will change ?
2. Change the activation functions of the first 3 layers to 'linear'. What will be the results ?
3. Bring back the activation functions to 'relu'
4. Change the training epochs to 5 and 5000 respectively. What will change ?
5. Does normalization help ?


### 5. Does normalization help ?

In [23]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
Xtr_scaled = scaler.fit_transform(Xtr)
Xtst_scaled = scaler.transform(Xtst)

print("Data normalized using StandardScaler.")
print(f"Original Xtr shape: {Xtr.shape}")
print(f"Scaled Xtr shape: {Xtr_scaled.shape}")

Data normalized using StandardScaler.
Original Xtr shape: (500, 8)
Scaled Xtr shape: (500, 8)


Now, let's redefine the model and train it with the normalized data.

In [24]:
# Redefine the keras model with relu activations (as per original problem, before linear experiment)
model_normalized = Sequential([
    keras.Input(shape=(8,)), # Define input shape using Input layer
    Dense(20, activation='relu'),
    Dense(20, activation='relu'),
    Dense(20, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the keras model
model_normalized.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Fit the keras model on the normalized training dataset
model_normalized.fit(Xtr_scaled, ytr, epochs=300, batch_size=10, verbose=0)

# Evaluate the keras model on the normalized test dataset
_, accuracy_normalized = model_normalized.evaluate(Xtst_scaled, ytst, verbose=0)
print('Accuracy with normalized data: %.2f ' % (accuracy_normalized*100))

Accuracy with normalized data: 74.91 


Let's also look at the classification report and confusion matrix for the model trained with normalized data.

In [25]:
# testing the model with normalized data
y_pred_normalized = model_normalized.predict(Xtst_scaled)
y_pred_normalized = np.round(y_pred_normalized)

print(confusion_matrix(ytst, y_pred_normalized))
print(classification_report(ytst, y_pred_normalized))

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
[[148  33]
 [ 34  52]]
              precision    recall  f1-score   support

         0.0       0.81      0.82      0.82       181
         1.0       0.61      0.60      0.61        86

    accuracy                           0.75       267
   macro avg       0.71      0.71      0.71       267
weighted avg       0.75      0.75      0.75       267

